In [ ]:
import json
from pathlib import Path

from datasets import Dataset


data_path = Path("data/finetuning_data_sft_prompt_completion.jsonl")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset file not found: {data_path}")

records = []
invalid_json = 0
missing_fields = 0
empty_prompt = 0
empty_completion = 0

with data_path.open("r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            invalid_json += 1
            continue

        prompt = str(row.get("prompt", "")).strip()
        completion = str(row.get("completion", "")).strip()

        if "prompt" not in row or "completion" not in row:
            missing_fields += 1
            continue
        if not prompt:
            empty_prompt += 1
            continue
        if not completion:
            empty_completion += 1
            continue

        records.append({"prompt": prompt, "completion": completion})

if not records:
    raise ValueError("No valid prompt/completion records found.")

# Deduplicate exact prompt-completion pairs.
seen = set()
unique_records = []
for r in records:
    key = (r["prompt"], r["completion"])
    if key not in seen:
        seen.add(key)
        unique_records.append(r)

duplicate_count = len(records) - len(unique_records)

# Build Dataset object.
dataset_raw = Dataset.from_list(unique_records)

# Split train/eval for reliable quality tracking during fine-tuning.
split = dataset_raw.train_test_split(test_size=0.1, seed=3407)
train_raw = split["train"]
eval_raw = split["test"]

# Format into Qwen chat style that your training loop can directly consume.
def to_qwen_chat(example):
    text = (
        f"<|im_start|>user\n{example['prompt']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['completion']}<|im_end|>"
    )
    return {"text": text}

train_dataset = train_raw.map(to_qwen_chat, remove_columns=train_raw.column_names)
eval_dataset = eval_raw.map(to_qwen_chat, remove_columns=eval_raw.column_names)

# Quality diagnostics.
prompt_lens = [len(r["prompt"]) for r in unique_records]
completion_lens = [len(r["completion"]) for r in unique_records]

def percentile(values, p):
    if not values:
        return 0
    values = sorted(values)
    idx = int((len(values) - 1) * p)
    return values[idx]

print("=== Dataset Loading & Cleaning ===")
print(f"Source file: {data_path}")
print(f"Valid rows before dedup: {len(records)}")
print(f"Duplicates removed: {duplicate_count}")
print(f"Final usable rows: {len(unique_records)}")
print(f"Invalid JSON rows: {invalid_json}")
print(f"Missing required fields: {missing_fields}")
print(f"Empty prompts removed: {empty_prompt}")
print(f"Empty completions removed: {empty_completion}")

print("\n=== Split ===")
print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")

print("\n=== Length Stats (characters) ===")
print(f"Prompt avg: {sum(prompt_lens)/len(prompt_lens):.1f}")
print(f"Prompt p95: {percentile(prompt_lens, 0.95)}")
print(f"Completion avg: {sum(completion_lens)/len(completion_lens):.1f}")
print(f"Completion p95: {percentile(completion_lens, 0.95)}")

print("\n=== Quick Verdict ===")
if len(unique_records) < 200:
    print("Small dataset: useful for style/domain adaptation, limited generalization.")
elif len(unique_records) < 1000:
    print("Medium-small dataset: good for targeted domain fine-tuning, quality depends heavily on consistency.")
else:
    print("Solid dataset size for domain fine-tuning. Keep tracking eval loss to avoid overfitting.")

if duplicate_count > 0:
    print("Some duplication existed and was removed. This improves training signal quality.")

print("\nFormatted training sample:")
print(train_dataset[0]["text"][:800])
